# Notebook 03 — Anotasi Manual ACSC

| Item | Detail |
|---|---|
| **Penelitian** | Aspect Category Sentiment Classification of EWA App Reviews Using IndoBERT |
| **Fase CRISP-DM** | Phase 3 — Data Preparation (Sub-tahap 2: Anotasi Manual) |
| **Input** | `dataset_clean_ewa_acsc.csv` (output Notebook 02) |
| **Output** | `dataset_annotated_ewa_acsc.csv` (format standar ACSC) |
| **Referensi** | Pontiki et al. (2014) SemEval-2014 Task 4 — ACSC Annotation Standard |

---

## Mengapa Anotasi Manual?

ACSC memerlukan **ground truth labels** pada dua level:
1. **Kategori aspek** — dimensi layanan mana yang dibicarakan
2. **Polaritas sentimen** — positif, negatif, atau netral terhadap aspek tersebut

Tidak ada model pre-trained yang dapat menghasilkan label ini secara otomatis
untuk domain EWA Indonesia tanpa data berlabel terlebih dahulu.
Anotasi manual dengan panduan yang mengacu standar akademis internasional
menghasilkan ground truth yang valid untuk melatih dan mengevaluasi model.

## Format Dataset ACSC — Standar SemEval (Pontiki et al., 2014)

Format standar internasional ACSC menggunakan **satu baris per pasangan (kalimat × aspek)**.
Satu ulasan yang membahas beberapa aspek menghasilkan beberapa baris.
Ini bukan bug — ini adalah standar yang ditetapkan oleh SemEval-2014 Task 4 Subtask 4.

```
CONTOH FORMAT:
Ulasan: "pencairan cepat tapi biaya adminnya besar"

→ Baris 1: review_id=R001 | aspek=Kecepatan Pencairan | sentimen=positif
→ Baris 2: review_id=R001 | aspek=Biaya/Potongan      | sentimen=negatif

Teks ulasan DIULANG di setiap baris — sesuai standar ACSC.
```

## 5 Kategori Aspek Penelitian Ini

| Kode | Kategori Aspek | Cakupan Domain EWA |
|---|---|---|
| A1 | Kecepatan Pencairan | Kecepatan transfer dana EWA ke rekening |
| A2 | Biaya/Potongan | Biaya admin, bunga, potongan layanan EWA |
| A3 | Kemudahan Penggunaan | Antarmuka, navigasi, registrasi, login |
| A4 | Customer Service | Ketanggapan dan kualitas layanan pelanggan |
| A5 | Keandalan Sistem | Error, crash, gangguan server, bug |

**Justifikasi 5 kategori aspek:** Ditetapkan melalui dua tahap — (1) kajian literatur
dimensi kualitas layanan fintech dan EWA (ILO, 2025; Lux & Chung, 2023); (2) verifikasi
empiris melalui pembacaan 100 sampel ulasan dari setiap aplikasi untuk memastikan
setiap kategori memiliki representasi yang memadai dalam data.

## 3 Kelas Sentimen

| Kelas | Definisi | Contoh |
|---|---|---|
| `positif` | Opini menguntungkan terhadap aspek | "pencairan cepat" |
| `negatif` | Opini merugikan terhadap aspek | "cs tidak responsif" |
| `netral` | Faktual / sentimen lemah / campuran | "pencairannya lumayan" |

## Alur Notebook

```
STEP 0   Setup & Load Data
  ↓
STEP 1   Panduan Anotasi Formal (referensi SemEval-2014)
  ↓
STEP 2   Generate Template Anotasi
  ↓
  ⏸️  ANOTASI MANUAL di Google Sheets
  ↓
STEP 3   Import & Parsing Hasil Anotasi
  ↓
STEP 4   Validasi & Koreksi Label
  ↓
STEP 5   Flatten ke Format ACSC
  ↓
STEP 6   Statistik & Distribusi Dataset
  ↓
STEP 7   Simpan Output Final
```

---
## STEP 0: Setup & Load Data

In [1]:
import os, warnings
import pandas as pd
import numpy  as np
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH  = '/content/drive/MyDrive/TA_ACSC_EWA'
PREP_DIR   = os.path.join(BASE_PATH, '02_preprocessing', 'data')
ANNOT_DIR  = os.path.join(BASE_PATH, '03_anotasi', 'data')
os.makedirs(ANNOT_DIR, exist_ok=True)

# Input
CLEAN_FILE    = os.path.join(PREP_DIR,  'dataset_clean_ewa_acsc.csv')
# Template & hasil anotasi
TEMPLATE_FILE = os.path.join(ANNOT_DIR, 'template_anotasi_acsc.csv')
HASIL_FILE    = os.path.join(ANNOT_DIR, 'hasil_anotasi_acsc.csv')
# Output final format ACSC
OUTPUT_FILE   = os.path.join(ANNOT_DIR, 'dataset_annotated_ewa_acsc.csv')
# Log validasi
INVALID_FILE  = os.path.join(ANNOT_DIR, 'anotasi_invalid_log.csv')

# ═══════════════════════════════════════════════════════════════════════════
# KONSTANTA ACSC — sesuai definisi penelitian
# ═══════════════════════════════════════════════════════════════════════════
ASPECTS = [
    'Kecepatan Pencairan',
    'Biaya/Potongan',
    'Kemudahan Penggunaan',
    'Customer Service',
    'Keandalan Sistem',
]
SENTIMENTS = ['positif', 'negatif', 'netral']
APPS = [
    'VENTENY', 'Mekari Flex', 'Paywatch',
    'Wagely', 'GajiGesa', 'AyoKasbon'
]

# ── Load dataset bersih ───────────────────────────────────────────────────
assert os.path.exists(CLEAN_FILE), f'❌ Jalankan NB02 dulu! File tidak ada: {CLEAN_FILE}'
df_clean = pd.read_csv(CLEAN_FILE, encoding='utf-8-sig')

print('✅ Dataset bersih dimuat')
print('=' * 60)
print(f'  Total ulasan bersih        : {len(df_clean):,}')
print(f'  Kandidat anotasi (non-skip): {(~df_clean["skip_flag"]).sum():,}')
print(f'  Kandidat skip              : {df_clean["skip_flag"].sum():,}')
print()
print('Distribusi per aplikasi (non-skip):')
ns = df_clean[~df_clean['skip_flag']]
for app, cnt in ns.groupby('app_name')['review_id'].count().items():
    print(f'  {app:<15}: {cnt:>4,}')
print(f'  {"-"*25}')
print(f'  {"TOTAL":<15}: {len(ns):>4,}')

Mounted at /content/drive
✅ Dataset bersih dimuat
  Total ulasan bersih        : 3,018
  Kandidat anotasi (non-skip): 2,648
  Kandidat skip              : 370

Distribusi per aplikasi (non-skip):
  AyoKasbon      :   38
  GajiGesa       :  387
  Mekari Flex    :  653
  Paywatch       :  206
  VENTENY        :  629
  Wagely         :  735
  -------------------------
  TOTAL          : 2,648


---
## STEP 1: Panduan Anotasi Formal

### ⚠️ BACA SELURUH PANDUAN INI SEBELUM MULAI ANOTASI

---

### 1.1 Dasar Akademis Panduan Ini

Panduan anotasi ini mengadaptasi prinsip standar internasional dari:
- **Pontiki et al. (2014)** — SemEval-2014 Task 4: Aspect Based Sentiment Analysis
- Disesuaikan untuk domain **Earned Wage Access (EWA)** Indonesia

---

### 1.2 Prinsip Dasar Anotasi

**Prinsip 1 — Gunakan teks original sebagai referensi**  
Selalu gunakan `review_text_original` — bukan `review_text_clean` — sebagai dasar
keputusan anotasi. Teks original lebih natural dan mencerminkan maksud pengguna.

**Prinsip 2 — Anotasi aspek yang DISEBUTKAN, bukan yang diasumsikan**  
Hanya anotasi aspek yang ada sinyal eksplisit atau implisit kuat dalam teks.
Jangan menambahkan aspek yang mungkin relevan tapi tidak disinggung.

**Prinsip 3 — Satu ulasan boleh memiliki 1–5 aspek**  
Isi `aspek_1` hingga `aspek_5` sesuai jumlah aspek yang ditemukan.
Tidak perlu mengisi semua slot — sisakan kosong jika tidak ada aspek lebih lanjut.

**Prinsip 4 — Setiap aspek harus disertai sentimen**  
Setiap `aspek_N` yang diisi HARUS memiliki `sentimen_N` yang sesuai.

**Prinsip 5 — Skip hanya jika tidak ada aspek sama sekali**  
Isi `skip=Y` HANYA jika tidak ada satu pun aspek yang dapat diidentifikasi.
Jika ada aspek → anotasi, jangan skip.

---

### 1.3 Definisi Kategori Aspek & Sinyal Kata Kunci

#### A1 — Kecepatan Pencairan
*Cakupan: kecepatan transfer, proses pencairan, waktu dana masuk rekening*

| Sentimen | Sinyal Kata Kunci | Contoh Ulasan |
|---|---|---|
| **positif** | cepat, instan, langsung cair, menit, segera masuk | "cair 5 menit, sangat cepat" |
| **negatif** | lambat, lama, belum cair, pending, gagal, tidak bisa tarik | "sudah 3 hari belum masuk rekening" |
| **netral** | lumayan, cukup cepat, standar, biasa | "pencairannya lumayan, tidak terlalu cepat" |

#### A2 — Biaya/Potongan
*Cakupan: biaya admin, biaya layanan, potongan gaji, bunga, administrasi*

| Sentimen | Sinyal Kata Kunci | Contoh Ulasan |
|---|---|---|
| **positif** | murah, gratis, wajar, terjangkau, worth it, tidak mahal | "biayanya masih wajar untuk layanan ini" |
| **negatif** | mahal, besar, mencekik, potongan besar, rugi, tidak worth | "potongan 30rb untuk 500rb, kemahalan" |
| **netral** | standar, lumayan, biasa, tidak terlalu mahal/murah | "biayanya lumayan standar" |

#### A3 — Kemudahan Penggunaan
*Cakupan: tampilan aplikasi, navigasi, proses daftar, login, fitur, UX*

| Sentimen | Sinyal Kata Kunci | Contoh Ulasan |
|---|---|---|
| **positif** | mudah, gampang, simpel, user friendly, intuitif, praktis | "tampilan simpel, mudah dipahami" |
| **negatif** | ribet, susah, bingung, tidak bisa login, tidak bisa daftar, lambat | "susah banget daftarnya, banyak tahapan" |
| **netral** | lumayan mudah, cukup simpel, biasa | "tampilannya cukup oke, tidak terlalu susah" |

#### A4 — Customer Service
*Cakupan: CS, admin, HR (dalam konteks menghubungi support), layanan pelanggan*

| Sentimen | Sinyal Kata Kunci | Contoh Ulasan |
|---|---|---|
| **positif** | responsif, cepat tanggap, membantu, solutif, ramah | "cs-nya responsif dan solutif" |
| **negatif** | tidak responsif, tidak dibalas, lama, diabaikan, tidak ada solusi | "cs tidak pernah balas chat" |
| **netral** | akhirnya dibalas, lumayan, cukup membantu | "cs lambat tapi akhirnya terselesaikan" |

#### A5 — Keandalan Sistem
*Cakupan: error, crash, bug, loading, server down, maintenance, gangguan teknis*

| Sentimen | Sinyal Kata Kunci | Contoh Ulasan |
|---|---|---|
| **positif** | stabil, tidak pernah error, lancar, berjalan baik | "sistemnya stabil, jarang ada masalah" |
| **negatif** | error, crash, bug, loading lama, tidak bisa buka, sering gangguan | "sering error pas mau tarik gaji" |
| **netral** | kadang error, sesekali lambat | "sesekali loading lama tapi biasanya oke" |

---

### 1.4 Kapan SKIP?

| Kondisi | Contoh | Keputusan |
|---|---|---|
| Terlalu umum, tidak ada aspek jelas | "mantap", "ok", "bagus" | SKIP → `skip=Y` |
| Hanya angka/simbol/emoji | "5", "👍", "💯" | SKIP |
| Off-topic | "salah download" | SKIP |
| Kalimat generik tanpa konteks | "sangat membantu" (tanpa konteks lebih lanjut) | SKIP |
| Ada aspek → jangan skip | "pencairan lambat sekali" | ANOTASI |
| Implisit tapi jelas | "sudah 3 hari belum cair" | ANOTASI (Kecepatan Pencairan, negatif) |

---

### 1.5 Kasus Ambigu & Panduan Penyelesaian

| Ulasan | Aspek | Sentimen | Justifikasi |
|---|---|---|---|
| "sangat membantu saat butuh dana darurat" | Kemudahan Penggunaan | positif | 'membantu' + EWA → kemudahan akses dana |
| "aplikasinya bagus" | Kemudahan Penggunaan | positif | penilaian aplikasi umum → UX |
| "pencairannya oke lah" | Kecepatan Pencairan | netral | 'oke lah' tidak tegas positif/negatif |
| "sudah cair" | Kecepatan Pencairan | netral | Pernyataan faktual tanpa sentimen |
| "cs-nya kurang" | Customer Service | negatif | 'kurang' = polarity negatif lemah |
| "bagus" (tanpa konteks) | — | — | SKIP |
| "limitnya kecil" | Biaya/Potongan | negatif | Limit kecil → dirugikan |
| "harus hubungi HR" | Customer Service | netral/negatif | Sesuai konteks kalimat |

---

### 1.6 Aturan Konsistensi (Self-Consistency)

Untuk menjaga reliabilitas anotasi oleh satu anotator:
1. Anotasi **per sesi per aplikasi** — jangan berpindah aplikasi dalam satu sesi
2. Setiap sesi dimulai dengan **review ulang 5–10 baris terakhir** sesi sebelumnya
3. Untuk kasus ambigu, **tulis catatan singkat** di kolom `catatan` (jika ada)
4. Gunakan **tabel Kasus Ambigu di atas** sebagai patokan konsisten

In [ ]:
# ── Cetak ringkasan panduan anotasi sebagai referensi cepat ──────────────
print('REFERENSI CEPAT PANDUAN ANOTASI')
print('=' * 60)
print()
print('5 KATEGORI ASPEK (isi persis seperti ini di spreadsheet):')
for i, asp in enumerate(ASPECTS, 1):
    print(f'  [{i}] {asp}')
print()
print('3 KELAS SENTIMEN:')
for sent in SENTIMENTS:
    print(f'  • {sent}')
print()
print('SKIP: isi kolom skip = Y')
print()
print('ATURAN KRITIS:')
print('  1. Gunakan review_text_original sebagai referensi')
print('  2. Jangan ubah review_id (primary key)')
print('  3. Aspek dan sentimen harus PERSIS sama seperti daftar di atas')
print('  4. Satu ulasan bisa punya 1–5 aspek')
print('  5. Skip=Y hanya jika TIDAK ADA aspek sama sekali')

REFERENSI CEPAT PANDUAN ANOTASI

5 KATEGORI ASPEK (isi persis seperti ini di spreadsheet):
  [1] Kecepatan Pencairan
  [2] Biaya/Potongan
  [3] Kemudahan Penggunaan
  [4] Customer Service
  [5] Keandalan Sistem

3 KELAS SENTIMEN:
  • positif
  • negatif
  • netral

SKIP: isi kolom skip = Y

ATURAN KRITIS:
  1. Gunakan review_text_original sebagai referensi
  2. Jangan ubah review_id (primary key)
  3. Aspek dan sentimen harus PERSIS sama seperti daftar di atas
  4. Satu ulasan bisa punya 1–5 aspek
  5. Skip=Y hanya jika TIDAK ADA aspek sama sekali


---
## STEP 2: Generate Template Anotasi

Cell ini membuat template CSV yang di-upload ke Google Sheets untuk diisi secara manual.

### Instruksi Google Sheets setelah upload:
1. **Data → Data Validation** pada kolom `aspek_1` s.d. `aspek_5`:
   - List: `Kecepatan Pencairan,Biaya/Potongan,Kemudahan Penggunaan,Customer Service,Keandalan Sistem`
2. **Data Validation** pada kolom `sentimen_1` s.d. `sentimen_5`:
   - List: `positif,negatif,netral`
3. **Data Validation** pada kolom `skip`:
   - List: `Y`
4. **JANGAN** ubah atau hapus kolom `review_id`
5. Simpan sebagai `hasil_anotasi_acsc.csv` dengan **delimiter titik koma (;)**

In [ ]:
# ── Generate template anotasi dari dataset bersih ────────────────────────

# Kolom yang ditampilkan ke anotator
df_template = df_clean[[
    'review_id', 'app_name', 'rating',
    'review_text_original', 'review_text_clean'
]].copy()

# Tambahkan nomor urut
df_template.insert(0, 'no', range(1, len(df_template) + 1))

# Kolom anotasi (diisi manual) — support hingga 5 aspek per ulasan
for slot in ['1','2','3','4','5']:
    df_template[f'aspek_{slot}']    = ''
    df_template[f'sentimen_{slot}'] = ''
df_template['skip'] = ''

# Urutkan: app_name → rating ascending (rating rendah = potensial negatif, prioritas)
df_template = df_template.sort_values(
    ['app_name', 'rating'], ascending=[True, True]
).reset_index(drop=True)
df_template['no'] = range(1, len(df_template) + 1)

# Simpan dengan delimiter titik koma
df_template.to_csv(TEMPLATE_FILE, index=False, encoding='utf-8-sig', sep=';')

print('✅ Template anotasi berhasil dibuat!')
print(f'   Path    : {TEMPLATE_FILE}')
print(f'   Total   : {len(df_template):,} baris')
print(f'   Kolom   : {df_template.columns.tolist()}')
print()
print('Rincian per aplikasi:')
print(f"  {'Aplikasi':<15} {'Total':>6}  {'Non-Skip':>8}  {'Skip':>6}  Estimasi waktu")
print(f"  {'─'*55}")
total_jam = 0
for app in APPS:
    sub     = df_clean[df_clean['app_name'] == app]
    ns_cnt  = (~sub['skip_flag']).sum()
    sk_cnt  = sub['skip_flag'].sum()
    jam     = ns_cnt / 60
    total_jam += jam
    print(f"  {app:<15} {len(sub):>6,}  {ns_cnt:>8,}  {sk_cnt:>6,}  ~{jam:.1f} jam")
print(f"  {'─'*55}")
print(f"  {'TOTAL':<15} {len(df_template):>6,}  {(~df_clean['skip_flag']).sum():>8,}  {df_clean['skip_flag'].sum():>6,}  ~{total_jam:.0f} jam total")
print()
print('⚠️  Catatan: Ulasan dengan skip_flag=True sudah teridentifikasi otomatis.')
print('   Namun tetap disertakan di template agar anotator dapat melakukan review ulang.')

✅ Template anotasi berhasil dibuat!
   Path    : /content/drive/MyDrive/TA_ACSC_EWA/03_anotasi/data/template_anotasi_acsc.csv
   Total   : 3,018 baris
   Kolom   : ['no', 'review_id', 'app_name', 'rating', 'review_text_original', 'review_text_clean', 'aspek_1', 'sentimen_1', 'aspek_2', 'sentimen_2', 'aspek_3', 'sentimen_3', 'aspek_4', 'sentimen_4', 'aspek_5', 'sentimen_5', 'skip']

Rincian per aplikasi:
  Aplikasi         Total  Non-Skip    Skip  Estimasi waktu
  ───────────────────────────────────────────────────────
  VENTENY            653       629      24  ~10.5 jam
  Mekari Flex        795       653     142  ~10.9 jam
  Paywatch           241       206      35  ~3.4 jam
  Wagely             854       735     119  ~12.2 jam
  GajiGesa           423       387      36  ~6.5 jam
  AyoKasbon           52        38      14  ~0.6 jam
  ───────────────────────────────────────────────────────
  TOTAL            3,018     2,648     370  ~44 jam total

⚠️  Catatan: Ulasan dengan skip_flag=T

---
## ⏸️ PAUSE — LAKUKAN ANOTASI MANUAL DI GOOGLE SHEETS

1. Upload `template_anotasi_acsc.csv` ke Google Sheets
2. Buat dropdown sesuai instruksi di atas
3. Isi kolom `aspek_1` s.d. `aspek_5` dan `sentimen_1` s.d. `sentimen_5`
4. Isi `skip=Y` untuk ulasan yang tidak dapat dianotasi
5. **Setelah selesai:** Download sebagai CSV dengan **delimiter titik koma (;)**
6. Simpan sebagai `hasil_anotasi_acsc.csv`
7. Upload ke: `TA_ACSC_EWA/03_anotasi/data/`

Setelah upload selesai, lanjutkan ke **STEP 3**.

---

## STEP 3: Import & Parsing Hasil Anotasi

Cell ini memuat `hasil_anotasi_acsc.csv`

In [ ]:
# ── Import hasil anotasi dengan robust parsing ───────────────────────────
assert os.path.exists(HASIL_FILE), (
    f'❌ File hasil anotasi tidak ditemukan: {HASIL_FILE}\n'
    f'   Selesaikan anotasi di Google Sheets dan upload ke path tersebut.'
)

# Parsing dengan delimiter titik koma
df_hasil = pd.read_csv(
    HASIL_FILE,
    sep           = ';',
    encoding      = 'utf-8-sig',
    on_bad_lines  = 'skip',   # skip baris yang corrupt
    dtype         = str,      # semua kolom sebagai string untuk menghindari type error
)

# Hapus kolom unnamed yang muncul dari trailing delimiter
df_hasil = df_hasil.loc[:, ~df_hasil.columns.str.startswith('Unnamed')]

# Bersihkan whitespace di semua cell
for col in df_hasil.columns:
    df_hasil[col] = df_hasil[col].astype(str).str.strip()

# Ganti string 'nan' menjadi kosong
df_hasil = df_hasil.replace('nan', '')

print('Hasil anotasi berhasil dimuat:')
print(f'  Total baris  : {len(df_hasil):,}')
print(f'  Kolom        : {df_hasil.columns.tolist()}')
print()

# Verifikasi kolom wajib ada
required_cols = ['review_id', 'app_name', 'rating',
                 'review_text_original', 'review_text_clean',
                 'aspek_1', 'sentimen_1', 'skip']
missing = [c for c in required_cols if c not in df_hasil.columns]
if missing:
    raise ValueError(f'❌ Kolom wajib tidak ada: {missing}')
print('✅ Semua kolom wajib tersedia')

# Verifikasi review_id valid (bukan UUID tersasar ke kolom app_name)
import re
uuid_pattern = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')
invalid_app  = df_hasil[~df_hasil['app_name'].isin(APPS + [''])]
if len(invalid_app) > 0:
    print(f'\n⚠️  {len(invalid_app)} baris memiliki app_name tidak valid (kemungkinan parsing error)')
    print('   Baris ini akan dikeluarkan dari dataset.')
    df_hasil = df_hasil[df_hasil['app_name'].isin(APPS)].copy().reset_index(drop=True)
    print(f'   Tersisa setelah filter: {len(df_hasil):,} baris')

print(f'\nDistribusi per aplikasi:')
for app in APPS:
    cnt = (df_hasil['app_name'] == app).sum()
    print(f'  {app:<15}: {cnt:>4,}')

Hasil anotasi berhasil dimuat:
  Total baris  : 3,018
  Kolom        : ['no', 'review_id', 'app_name', 'rating', 'review_text_original', 'review_text_clean', 'aspek_1', 'sentimen_1', 'aspek_2', 'sentimen_2', 'aspek_3', 'sentimen_3', 'aspek_4', 'sentimen_4', 'aspek_5', 'sentimen_5', 'skip']

✅ Semua kolom wajib tersedia

Distribusi per aplikasi:
  VENTENY        :  653
  Mekari Flex    :  795
  Paywatch       :  241
  Wagely         :  854
  GajiGesa       :  423
  AyoKasbon      :   52


---
## STEP 4: Validasi & Koreksi Label

Validasi dilakukan untuk memastikan:
1. Semua nilai aspek adalah salah satu dari 5 kategori yang valid
2. Semua nilai sentimen adalah salah satu dari `positif`, `negatif`, `netral`
3. Tidak ada aspek yang terisi tanpa sentimen (atau sebaliknya)
4. Tidak ada inkonsistensi skip — baris skip=Y tidak boleh memiliki aspek terisi

In [ ]:
print('STEP 4: Validasi Label Anotasi')
print('=' * 60)

invalid_rows = []
n_fixed = 0

for idx, row in df_hasil.iterrows():
    skip_val = str(row.get('skip', '')).strip().upper()
    is_skip  = (skip_val == 'Y')

    for slot in ['1','2','3','4','5']:
        asp  = str(row.get(f'aspek_{slot}',    '')).strip()
        sent = str(row.get(f'sentimen_{slot}', '')).strip()

        if not asp:  # slot kosong → stop iterasi slot ini
            break

        # Masalah 1: skip=Y tapi aspek terisi
        if is_skip and asp:
            invalid_rows.append({
                'idx': idx, 'review_id': row.get('review_id',''),
                'app_name': row.get('app_name',''),
                'masalah': f'skip=Y tapi aspek_{slot} terisi',
                'nilai': asp
            })
            continue

        # Masalah 2: aspek tidak valid
        if asp not in ASPECTS:
            invalid_rows.append({
                'idx': idx, 'review_id': row.get('review_id',''),
                'app_name': row.get('app_name',''),
                'masalah': f'aspek_{slot} tidak valid',
                'nilai': asp[:60]
            })
            continue

        # Masalah 3: sentimen tidak valid atau kosong
        if sent not in SENTIMENTS:
            # Coba koreksi otomatis: sentimen di kolom aspek berikutnya (column shift)
            next_asp = str(row.get(f'aspek_{str(int(slot)+1)}', '')).strip()
            if sent in ASPECTS:
                invalid_rows.append({
                    'idx': idx, 'review_id': row.get('review_id',''),
                    'app_name': row.get('app_name',''),
                    'masalah': f'sentimen_{slot} berisi nama aspek (column shift)',
                    'nilai': sent
                })
            else:
                invalid_rows.append({
                    'idx': idx, 'review_id': row.get('review_id',''),
                    'app_name': row.get('app_name',''),
                    'masalah': f'sentimen_{slot} tidak valid: "{sent}"',
                    'nilai': sent
                })

# Simpan log invalid
df_invalid = pd.DataFrame(invalid_rows)
if len(df_invalid) > 0:
    df_invalid.to_csv(INVALID_FILE, index=False, encoding='utf-8-sig')

# Hitung statistik validasi
skip_cnt   = (df_hasil['skip'].str.upper() == 'Y').sum()
total_cnt  = len(df_hasil)
annot_cnt  = total_cnt - skip_cnt

# Hitung baris dengan aspek_1 valid
valid_cnt = (
    df_hasil['aspek_1'].isin(ASPECTS) &
    df_hasil['sentimen_1'].isin(SENTIMENTS) &
    (df_hasil['skip'].str.upper() != 'Y')
).sum()

print(f'Total baris hasil anotasi : {total_cnt:,}')
print(f'  Skip=Y                  : {skip_cnt:,}')
print(f'  Dianotasi               : {annot_cnt:,}')
print(f'  Valid (aspek+sent ok)   : {valid_cnt:,}')
print(f'  Bermasalah              : {len(df_invalid):,}')

if len(df_invalid) > 0:
    print(f'\n⚠️  {len(df_invalid)} baris bermasalah — simpan ke: {INVALID_FILE}')
    print(   '   Baris bermasalah akan DIKELUARKAN dari dataset final.')
    # Tampilkan ringkasan masalah
    print()
    for masalah, cnt in df_invalid['masalah'].value_counts().items():
        print(f'   {masalah}: {cnt}')
else:
    print('\n✅ Tidak ada masalah label ditemukan!')

STEP 4: Validasi Label Anotasi
Total baris hasil anotasi : 3,018
  Skip=Y                  : 943
  Dianotasi               : 2,075
  Valid (aspek+sent ok)   : 2,075
  Bermasalah              : 0

✅ Tidak ada masalah label ditemukan!


In [ ]:
# ── Hapus baris invalid dari dataset sebelum flatten ─────────────────────
if len(df_invalid) > 0:
    invalid_idx = set(df_invalid['idx'].tolist())
    # Catatan: satu baris bisa memiliki beberapa masalah (terhitung beberapa kali)
    # Kita hanya drop baris yang seluruh anotasinya tidak valid
    # Baris yang hanya sebagian slot invalid → aspek valid tetap diproses
    print(f'Baris dengan masalah: {len(invalid_idx)} baris unik')
    print('Baris ini tetap diproses pada STEP 5 — slot invalid akan diabaikan')
    print('(hanya baris yang aspek_1 + sentimen_1 keduanya valid yang masuk output)')

# Konfirmasi distribusi per aplikasi setelah pembersihan
print()
print('Distribusi valid per aplikasi (pre-flatten):')
valid_mask = (
    df_hasil['aspek_1'].isin(ASPECTS) &
    df_hasil['sentimen_1'].isin(SENTIMENTS) &
    (df_hasil['skip'].str.upper() != 'Y')
)
for app in APPS:
    cnt = (valid_mask & (df_hasil['app_name'] == app)).sum()
    print(f'  {app:<15}: {cnt:>4,}')
print(f'  {"-"*25}')
print(f'  {"TOTAL":<15}: {valid_mask.sum():>4,}')


Distribusi valid per aplikasi (pre-flatten):
  VENTENY        :  540
  Mekari Flex    :  525
  Paywatch       :  172
  Wagely         :  519
  GajiGesa       :  286
  AyoKasbon      :   33
  -------------------------
  TOTAL          : 2,075


---
## STEP 5: Flatten ke Format ACSC

Konversi dari format wide `(aspek_1..5, sentimen_1..5)` ke format long
standar ACSC `(1 baris per pasangan review × aspek)`.

In [ ]:
# ── Flatten ke format ACSC ────────────────────────────────────────────────
acsc_rows = []
n_skipped = 0
n_invalid_slot = 0

for _, row in df_hasil.iterrows():

    # 1. Lewati baris skip=Y
    if str(row.get('skip', '')).strip().upper() == 'Y':
        n_skipped += 1
        continue

    # 2. Iterasi setiap slot aspek (1–5)
    for slot in ['1','2','3','4','5']:
        asp  = str(row.get(f'aspek_{slot}',    '')).strip()
        sent = str(row.get(f'sentimen_{slot}', '')).strip()

        # Stop jika slot kosong
        if not asp:
            break

        # Skip slot yang tidak valid
        if asp not in ASPECTS or sent not in SENTIMENTS:
            n_invalid_slot += 1
            continue

        # Tambahkan ke dataset ACSC
        acsc_rows.append({
            'review_id'           : str(row.get('review_id',            '')).strip(),
            'app_name'            : str(row.get('app_name',             '')).strip(),
            'rating'              : row.get('rating', ''),
            'review_date'         : str(row.get('review_date',          '')).strip(),
            'review_text_original': str(row.get('review_text_original', '')).strip(),
            'review_text_clean'   : str(row.get('review_text_clean',    '')).strip(),
            'aspect_category'     : asp,
            'sentiment'           : sent,
        })

df_acsc = pd.DataFrame(acsc_rows)

# Konversi rating ke integer
df_acsc['rating'] = pd.to_numeric(df_acsc['rating'], errors='coerce').fillna(3).astype(int)

# Urutkan
df_acsc = df_acsc.sort_values(
    ['app_name','aspect_category','sentiment']
).reset_index(drop=True)

print('✅ Flatten selesai!')
print(f'   Total baris ACSC        : {len(df_acsc):,}')
print(f'   Baris skip diabaikan    : {n_skipped:,}')
print(f'   Slot invalid diabaikan  : {n_invalid_slot:,}')
print(f'   Kolom                   : {df_acsc.columns.tolist()}')
print()

# Preview
print('Contoh 5 baris (format ACSC):')
try:
    from IPython.display import display
    display(df_acsc[[
        'review_id','app_name','aspect_category',
        'sentiment','review_text_original'
    ]].head(5))
except:
    print(df_acsc[[
        'app_name','aspect_category','sentiment','review_text_original'
    ]].head(5).to_string())

# Verifikasi multi-aspek
print()
print('Distribusi jumlah aspek per ulasan:')
rid_counts = df_acsc.groupby('review_id').size().value_counts().sort_index()
for n_asp, cnt in rid_counts.items():
    print(f'  {n_asp} aspek : {cnt:,} ulasan')
n_unique_reviews = df_acsc['review_id'].nunique()
print(f'  Total ulasan unik yang teranotasi: {n_unique_reviews:,}')
print(f'  Rata-rata aspek per ulasan       : {len(df_acsc)/n_unique_reviews:.2f}')

✅ Flatten selesai!
   Total baris ACSC        : 2,311
   Baris skip diabaikan    : 943
   Slot invalid diabaikan  : 0
   Kolom                   : ['review_id', 'app_name', 'rating', 'review_date', 'review_text_original', 'review_text_clean', 'aspect_category', 'sentiment']

Contoh 5 baris (format ACSC):


,review_id,app_name,aspect_category,sentiment,review_text_original
0,b9c54e11-a4cc-4c18-bf2b-6595ca110cec,AyoKasbon,Biaya/Potongan,netral,biaya admin berbeda tergantung jumlah yang dit...
1,9d0176a5-70e6-4005-8918-d8f7f1702f3b,AyoKasbon,Customer Service,negatif,tolong batalkan pendaftaran saya dan jangan hu...
2,a56d3668-9e21-4709-833b-30a06cd61546,AyoKasbon,Keandalan Sistem,negatif,Wah ini ilegal selesai isi formulir mau masuk ...
3,3fd04ea4-c633-49e7-b8bf-0c086a18e7ef,AyoKasbon,Keandalan Sistem,negatif,lol mau daftar aja gak bisa sandi salah baru k...
4,9e120517-b36f-4c73-9177-94b1f04f36e8,AyoKasbon,Keandalan Sistem,negatif,"apk tolol , apk bodoh yg buat bodoh , apk pend..."



Distribusi jumlah aspek per ulasan:
  1 aspek : 1,863 ulasan
  2 aspek : 191 ulasan
  3 aspek : 19 ulasan
  4 aspek : 1 ulasan
  5 aspek : 1 ulasan
  Total ulasan unik yang teranotasi: 2,075
  Rata-rata aspek per ulasan       : 1.11


---
## STEP 6: Statistik & Distribusi Dataset ACSC

Statistik ini menjadi data resmi yang dilaporkan di Bab III dan Bab IV dokumen TA.

In [ ]:
# ── Tabel distribusi aspek × sentimen ────────────────────────────────────
print('DISTRIBUSI DATASET ACSC — SUMBER ANGKA RESMI DOKUMEN TA')
print('=' * 70)

ct = pd.crosstab(
    df_acsc['aspect_category'],
    df_acsc['sentiment'],
    margins=True, margins_name='Total'
)
# Urutan kolom dan baris
col_order = [c for c in ['positif','negatif','netral','Total'] if c in ct.columns]
row_order = [r for r in ASPECTS + ['Total'] if r in ct.index]
ct = ct.reindex(index=row_order, columns=col_order, fill_value=0)

print()
print(ct.to_string())
print()

DISTRIBUSI DATASET ACSC — SUMBER ANGKA RESMI DOKUMEN TA

sentiment             positif  negatif  netral  Total
aspect_category                                      
Kecepatan Pencairan        93      169      35    297
Biaya/Potongan             51      118      27    196
Kemudahan Penggunaan      973      248      19   1240
Customer Service           22       62      29    113
Keandalan Sistem           67      375      23    465
Total                    1206      972     133   2311



In [ ]:
# ── Analisis class imbalance ──────────────────────────────────────────────
print('ANALISIS KETIDAKSEIMBANGAN KELAS')
print('=' * 70)
print()
print(f"  {'Aspek':<22} {'Sentimen':<12} {'Count':>6}  Status")
print(f"  {'─'*60}")

for asp in ASPECTS:
    for sent in SENTIMENTS:
        try:
            n = ct.loc[asp, sent]
        except KeyError:
            n = 0
        if n < 30:
            status = '🔴 Sangat sedikit (<30)'
        elif n < 50:
            status = '🟡 Sedikit (30–50)'
        elif n < 100:
            status = '🟢 Cukup (50–100)'
        else:
            status = '✅ Baik (>100)'
        print(f'  {asp:<22} {sent:<12} {n:>6}  {status}')
    print()

print()
print('Catatan: class imbalance yang signifikan diatasi melalui:')
print('  1. Class weighting pada loss function saat training')
print('  2. Evaluasi menggunakan Macro-F1 (memberikan bobot sama ke semua kelas)')

ANALISIS KETIDAKSEIMBANGAN KELAS

  Aspek                  Sentimen      Count  Status
  ────────────────────────────────────────────────────────────
  Kecepatan Pencairan    positif          93  🟢 Cukup (50–100)
  Kecepatan Pencairan    negatif         169  ✅ Baik (>100)
  Kecepatan Pencairan    netral           35  🟡 Sedikit (30–50)

  Biaya/Potongan         positif          51  🟢 Cukup (50–100)
  Biaya/Potongan         negatif         118  ✅ Baik (>100)
  Biaya/Potongan         netral           27  🔴 Sangat sedikit (<30)

  Kemudahan Penggunaan   positif         973  ✅ Baik (>100)
  Kemudahan Penggunaan   negatif         248  ✅ Baik (>100)
  Kemudahan Penggunaan   netral           19  🔴 Sangat sedikit (<30)

  Customer Service       positif          22  🔴 Sangat sedikit (<30)
  Customer Service       negatif          62  🟢 Cukup (50–100)
  Customer Service       netral           29  🔴 Sangat sedikit (<30)

  Keandalan Sistem       positif          67  🟢 Cukup (50–100)
  Keandalan Si

In [ ]:
# ── Distribusi per aplikasi ───────────────────────────────────────────────
print('DISTRIBUSI PER APLIKASI EWA')
print('=' * 65)
print()
for app in APPS:
    sub = df_acsc[df_acsc['app_name'] == app]
    if sub.empty:
        print(f'  {app:<15} : tidak ada data')
        continue
    n_pos  = (sub['sentiment'] == 'positif').sum()
    n_neg  = (sub['sentiment'] == 'negatif').sum()
    n_net  = (sub['sentiment'] == 'netral').sum()
    n_rev  = sub['review_id'].nunique()
    print(f'  {app:<15} | {len(sub):>4} baris | {n_rev:>4} ulasan | '
          f'Pos:{n_pos:>4} ({n_pos/len(sub)*100:.0f}%) '
          f'Neg:{n_neg:>4} ({n_neg/len(sub)*100:.0f}%) '
          f'Net:{n_net:>4} ({n_net/len(sub)*100:.0f}%)')
print()
print(f'  TOTAL: {len(df_acsc):,} baris | {df_acsc["review_id"].nunique():,} ulasan unik')

DISTRIBUSI PER APLIKASI EWA

  VENTENY         |  591 baris |  540 ulasan | Pos: 398 (67%) Neg: 168 (28%) Net:  25 (4%)
  Mekari Flex     |  586 baris |  525 ulasan | Pos: 298 (51%) Neg: 265 (45%) Net:  23 (4%)
  Paywatch        |  177 baris |  172 ulasan | Pos: 125 (71%) Neg:  30 (17%) Net:  22 (12%)
  Wagely          |  590 baris |  519 ulasan | Pos: 234 (40%) Neg: 324 (55%) Net:  32 (5%)
  GajiGesa        |  331 baris |  286 ulasan | Pos: 139 (42%) Neg: 163 (49%) Net:  29 (9%)
  AyoKasbon       |   36 baris |   33 ulasan | Pos:  12 (33%) Neg:  22 (61%) Net:   2 (6%)

  TOTAL: 2,311 baris | 2,075 ulasan unik


In [ ]:
# ── Distribusi per aspek (untuk Bab IV grafik) ────────────────────────────
print('DISTRIBUSI PER ASPEK (untuk grafik Bab IV)')
print('=' * 60)
print()
for asp in ASPECTS:
    sub  = df_acsc[df_acsc['aspect_category'] == asp]
    pct  = len(sub) / len(df_acsc) * 100
    bar  = '█' * max(1, int(pct / 2))
    print(f'  {asp:<22}: {len(sub):>5,} ({pct:.1f}%) {bar}')
print()

# Distribusi sentimen keseluruhan
print('DISTRIBUSI SENTIMEN KESELURUHAN')
print('=' * 60)
for sent in SENTIMENTS:
    n   = (df_acsc['sentiment'] == sent).sum()
    pct = n / len(df_acsc) * 100
    bar = '█' * max(1, int(pct / 2))
    print(f'  {sent:<10}: {n:>5,} ({pct:.1f}%) {bar}')
print()

# Analisis rating vs sentimen (validasi proxy)
print('VALIDASI PROXY: Rating vs Sentimen Anotasi')
print('=' * 60)
print('(Distribusi sentimen per rating bintang — uji konsistensi anotasi)')
rt = pd.crosstab(df_acsc['rating'], df_acsc['sentiment'])
col_order = [c for c in ['positif','negatif','netral'] if c in rt.columns]
rt = rt.reindex(columns=col_order, fill_value=0)
print(rt.to_string())
print()
print('Ekspektasi: ⭐1-2 mayoritas negatif, ⭐4-5 mayoritas positif')

DISTRIBUSI PER ASPEK (untuk grafik Bab IV)

  Kecepatan Pencairan   :   297 (12.9%) ██████
  Biaya/Potongan        :   196 (8.5%) ████
  Kemudahan Penggunaan  : 1,240 (53.7%) ██████████████████████████
  Customer Service      :   113 (4.9%) ██
  Keandalan Sistem      :   465 (20.1%) ██████████

DISTRIBUSI SENTIMEN KESELURUHAN
  positif   : 1,206 (52.2%) ██████████████████████████
  negatif   :   972 (42.1%) █████████████████████
  netral    :   133 (5.8%) ██

VALIDASI PROXY: Rating vs Sentimen Anotasi
(Distribusi sentimen per rating bintang — uji konsistensi anotasi)
sentiment  positif  negatif  netral
rating                             
1               23      590      13
2                5       83       1
3                5       68      21
4               69       37      12
5             1104      194      86

Ekspektasi: ⭐1-2 mayoritas negatif, ⭐4-5 mayoritas positif


---
## STEP 7: Simpan Output Final

In [ ]:
# ── Simpan dataset ACSC final ─────────────────────────────────────────────
df_acsc.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
size_kb = os.path.getsize(OUTPUT_FILE) / 1024

# Hitung statistik ringkasan
n_clean   = len(df_clean)
n_hasil   = len(df_hasil)
n_skip    = (df_hasil['skip'].str.upper() == 'Y').sum()
n_annot   = n_hasil - n_skip
n_invalid_log = len(df_invalid) if 'df_invalid' in dir() and len(df_invalid) > 0 else 0

print('=' * 65)
print('RINGKASAN ANOTASI ACSC — SUMBER ANGKA RESMI DOKUMEN TA')
print('=' * 65)
print(f'Dataset bersih (input)     : {n_clean:,} ulasan')
print(f'  - Non-skip (kandidat)    : {(~df_clean["skip_flag"]).sum():,}')
print(f'  - Skip (otomatis NB02)   : {df_clean["skip_flag"].sum():,}')
print()
print(f'Hasil anotasi (diproses)   : {n_hasil:,} baris')
print(f'  - Skip=Y (manual)        : {n_skip:,}')
print(f'  - Dianotasi              : {n_annot:,}')
print(f'  - Label invalid (log)    : {n_invalid_log:,}')
print()
print(f'Output dataset ACSC final  : {len(df_acsc):,} baris')
print(f'  - Ulasan unik teranotasi : {df_acsc["review_id"].nunique():,}')
print(f'  - Rata-rata aspek/ulasan : {len(df_acsc)/df_acsc["review_id"].nunique():.2f}')
print()
print('Format: 1 baris per pasangan (ulasan × aspek) — standar SemEval-2014 Task 4')
print()
print('Kolom output:')
for col in df_acsc.columns:
    desc = {
        'review_id'           : 'Primary key — UUID',
        'app_name'            : 'Nama aplikasi EWA',
        'rating'              : 'Rating bintang 1–5',
        'review_date'         : 'Tanggal ulasan',
        'review_text_original': 'Teks asli (referensi)',
        'review_text_clean'   : 'Teks bersih (input model)',
        'aspect_category'     : 'Kategori aspek → INPUT sentence-pair',
        'sentiment'           : 'Label sentimen → TARGET klasifikasi',
    }.get(col, '')
    print(f'  - {col:<25}: {desc}')
print()
print(f'Output: {OUTPUT_FILE}')
print(f'Ukuran: {size_kb:.1f} KB')
print()
print('=' * 65)
print('✅ NOTEBOOK 03 SELESAI!')
print('   → Lanjut: Notebook 04 — Baseline SVM + TF-IDF')
print('=' * 65)

RINGKASAN ANOTASI ACSC — SUMBER ANGKA RESMI DOKUMEN TA
Dataset bersih (input)     : 3,018 ulasan
  - Non-skip (kandidat)    : 2,648
  - Skip (otomatis NB02)   : 370

Hasil anotasi (diproses)   : 3,018 baris
  - Skip=Y (manual)        : 943
  - Dianotasi              : 2,075
  - Label invalid (log)    : 0

Output dataset ACSC final  : 2,311 baris
  - Ulasan unik teranotasi : 2,075
  - Rata-rata aspek/ulasan : 1.11

Format: 1 baris per pasangan (ulasan × aspek) — standar SemEval-2014 Task 4

Kolom output:
  - review_id                : Primary key — UUID
  - app_name                 : Nama aplikasi EWA
  - rating                   : Rating bintang 1–5
  - review_date              : Tanggal ulasan
  - review_text_original     : Teks asli (referensi)
  - review_text_clean        : Teks bersih (input model)
  - aspect_category          : Kategori aspek → INPUT sentence-pair
  - sentiment                : Label sentimen → TARGET klasifikasi

Output: /content/drive/MyDrive/TA_ACSC_EWA/03_anot

---
## STEP 8: Finalisasi Dataset — Dedup Pasca-Normalisasi & Perbaikan Konflik

Setelah normalisasi NB02, sejumlah ulasan berbeda menjadi berteks bersih identik (misalnya "Bagus", "bagus", "baGus"). Deduplication tahap kedua dilakukan pada LEVEL ULASAN (review_id) agar baris multi-aspek tidak kolaps: satu baris per review_id diambil, teks bersih dideduplikasi, lalu seluruh baris ACSC difilter ke review_id yang dipertahankan. Ulasan yang dibuang didokumentasikan pada audit_near_duplicate_tahap2.csv. Setelah itu satu konflik label diselesaikan.

In [3]:
# ══════════════════════════════════════════════════════════════════════
# SEL BARU NB03-1 — Dedup review_text_clean di level REVIEW  [Revisi C1]
# Mandiri: memuat anotasi tersimpan, TIDAK re-flatten.
# ══════════════════════════════════════════════════════════════════════
import pandas as pd

ANN_PATH = "/content/drive/MyDrive/TA_ACSC_EWA/03_anotasi/data/dataset_annotated_ewa_acsc.csv"  # ← sesuaikan
ann = pd.read_csv(ANN_PATH)
print(f"Baris awal        : {len(ann)}  | ulasan unik: {ann['review_id'].nunique()}")

# Dedup di level REVIEW (bukan level baris) agar multi-aspek TIDAK kolaps:
#   1) ambil 1 baris per review_id + teksnya
#   2) dedup pada review_text_clean (keep first) → daftar review_id dipertahankan
#   3) filter SEMUA baris ke review_id itu (semua aspek ikut)
reviews  = ann[['review_id', 'review_text_clean']].drop_duplicates('review_id').sort_values('review_id')
keep_ids = set(reviews.drop_duplicates(subset=['review_text_clean'], keep='first')['review_id'])
drop_ids = set(reviews['review_id']) - keep_ids

ann_dedup = ann[ann['review_id'].isin(keep_ids)].copy()
print(f"Ulasan teks-duplikat dibuang : {len(drop_ids)}")
print(f"Baris ACSC        : {len(ann)} → {len(ann_dedup)} (−{len(ann)-len(ann_dedup)})")
print("\nContoh ulasan yang dibuang (teks identik, review_id beda):")
for t in ann[ann['review_id'].isin(list(drop_ids)[:5])]['review_text_clean'].head(5):
    print(f"   '{t[:48]}'")

Baris awal        : 2311  | ulasan unik: 2075
Ulasan teks-duplikat dibuang : 44
Baris ACSC        : 2311 → 2267 (−44)

Contoh ulasan yang dibuang (teks identik, review_id beda):
   'sangat membantu terima kasih'
   'sangat membantu sekali'
   'terima kasih sangat membantu'
   'membantu sekali'
   'aplikasi yang sangat membantu'


In [4]:
# Perbaikan konflik label: review_id 541f7937
# Ulasan "tercepat dikelasnya besarin dong limitnya..." terlabel Kecepatan Pencairan
# positif DAN negatif sekaligus. Berdasarkan panduan, "besarin limit" bukan evaluasi
# kecepatan, sedangkan "tercepat dikelasnya" adalah evaluasi positif kecepatan,
# sehingga baris negatif dihapus.

mask_konflik = (
    ann_dedup['review_id'].str.startswith('541f7937') &
    (ann_dedup['aspect_category'] == 'Kecepatan Pencairan') &
    (ann_dedup['sentiment'] == 'negatif')
)
print(f"Baris konflik (negatif keliru) dihapus: {mask_konflik.sum()}")
ann_final = ann_dedup[~mask_konflik].copy()

konflik_sisa = ann_final.groupby(['review_id', 'aspect_category'])['sentiment'].nunique()
n_konflik = (konflik_sisa > 1).sum()
print(f"Konflik (review_id, aspek) tersisa    : {n_konflik}")
assert n_konflik == 0, "❌ Masih ada konflik label!"
print(f"Baris setelah perbaikan               : {len(ann_final)}")

Baris konflik (negatif keliru) dihapus: 1
Konflik (review_id, aspek) tersisa    : 0
Baris setelah perbaikan               : 2266


In [6]:
print("═"*55); print("DATASET FINAL (dedup + fix konflik)"); print("═"*55)
print(f"Total baris ACSC : {len(ann_final)}")
print(f"Ulasan unik      : {ann_final['review_id'].nunique()}")
print("\nDistribusi sentimen:")
for s, n in ann_final['sentiment'].value_counts().items():
    print(f"  {s:<8}: {n:5d} ({n/len(ann_final)*100:4.1f}%)")
print("\nDistribusi aspek:")
for a, n in ann_final['aspect_category'].value_counts().items():
    print(f"  {a:<22}: {n:5d}")

OUT = "/content/drive/MyDrive/TA_ACSC_EWA/03_anotasi/data/dataset_annotated_final.csv"  
ann_final.to_csv(OUT, index=False)
print(f"\n✅ Disimpan: {OUT}")

═══════════════════════════════════════════════════════
DATASET FINAL (dedup + fix konflik)
═══════════════════════════════════════════════════════
Total baris ACSC : 2266
Ulasan unik      : 2031

Distribusi sentimen:
  positif :  1167 (51.5%)
  negatif :   967 (42.7%)
  netral  :   132 ( 5.8%)

Distribusi aspek:
  Kemudahan Penggunaan  :  1198
  Keandalan Sistem      :   464
  Kecepatan Pencairan   :   296
  Biaya/Potongan        :   196
  Customer Service      :   112

✅ Disimpan: /content/drive/MyDrive/TA_ACSC_EWA/03_anotasi/data/dataset_annotated_final.csv

